<a href="https://colab.research.google.com/github/Aleeha-Fatima-del/Internee.pk-projects/blob/main/Resume_Screening_Automation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Automated Resume Screening & Job Matching System

**Objective:** Automate resume filtering to match candidates with job openings.

**Pipeline:**
1. Upload resume (PDF / DOCX / TXT)
2. Extract skills & experience with **spaCy** + **NLTK**
3. Structure the resume into clean JSON
4. Match against job descriptions using **TF-IDF** and **BERT** (sentence-transformers)
5. Search **real, live job openings** (free public API) and rank them against your profile




## 1. Install & Import Dependencies

In [ ]:
!pip install -q spacy nltk scikit-learn sentence-transformers PyPDF2 python-docx pandas requests
!python -m spacy download en_core_web_sm -q

import spacy
import nltk
import re
import json
import io
import requests
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nlp = spacy.load("en_core_web_sm")
STOPWORDS = set(stopwords.words('english'))

print("✅ All libraries loaded.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All libraries loaded.


## 2. Upload Your Resume
Supports **PDF**, **DOCX**, and **TXT**.

In [ ]:
from google.colab import files
import PyPDF2
import docx

uploaded = files.upload()
resume_filename = list(uploaded.keys())[0]

def extract_text(filename):
    if filename.lower().endswith('.pdf'):
        reader = PyPDF2.PdfReader(io.BytesIO(uploaded[filename]))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif filename.lower().endswith('.docx'):
        doc = docx.Document(io.BytesIO(uploaded[filename]))
        return "\n".join(p.text for p in doc.paragraphs)
    else:
        return uploaded[filename].decode('utf-8', errors='ignore')

resume_text = extract_text(resume_filename)
print(f"✅ Extracted {len(resume_text)} characters from {resume_filename}")
print("\n--- Preview ---\n")
print(resume_text[:600])


Saving Aleeha_Fatima_CV.pdf to Aleeha_Fatima_CV.pdf
✅ Extracted 1519 characters from Aleeha_Fatima_CV.pdf

--- Preview ---

AF
Aleeha Fatima
DATA SCIENCE STUDENT
CONTACT
@
fatimaaleeha054l@gmail.com
T
+92 315 3937296
P
Sahiwal, Pakistan
SKILLS
PROGRAMMING
C++
Python
Java
WEB & TOOLS
HTML
CSS
Machine Learning
MS Office
SOFT SKILLS
Teamwork
Problem Solving
Time Management
LANGUAGES
Urdu
Native
English
Intermediate
COURSEWORK
Programming Fundamentals
Data Structures
Object Oriented Programming
Data Science
Machine Learning
Discrete Mathematics
REFERENCES
Available upon request
Aleeha 
Fatima
BS Mathematics with Data Science  ·  4th Semester
CAREER OBJECTIVE
Motivated and detail-oriented Data Science student currently 


## 3. Skills Taxonomy
A curated skills list used for matching (edit/extend freely — add skills relevant to your field).

In [ ]:
SKILLS_DB = [
    # Programming languages
    "python", "java", "c++", "c", "javascript", "typescript", "sql", "r", "matlab", "html", "css",
    # Data science / ML
    "machine learning", "deep learning", "nlp", "natural language processing", "computer vision",
    "data analysis", "data science", "data visualization", "statistics", "pandas", "numpy",
    "scikit-learn", "tensorflow", "pytorch", "keras", "spacy", "nltk", "opencv", "tableau",
    "power bi", "excel", "regression", "classification", "clustering", "neural networks",
    "random forest", "xgboost", "time series analysis", "a/b testing", "big data", "hadoop", "spark",
    # Web / software
    "react", "node.js", "django", "flask", "rest api", "git", "github", "docker", "kubernetes",
    "aws", "azure", "gcp", "linux", "agile", "scrum", "ci/cd",
    # Databases
    "mysql", "postgresql", "mongodb", "firebase", "sqlite",
    # Soft/business skills
    "project management", "communication", "leadership", "teamwork", "problem solving",
    "critical thinking", "presentation", "content strategy", "social media marketing",
    "digital marketing", "seo", "google analytics", "meta graph api",
]

from spacy.matcher import PhraseMatcher
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(skill) for skill in SKILLS_DB]
matcher.add("SKILLS", patterns)
print(f"✅ Skills taxonomy loaded: {len(SKILLS_DB)} skills")


✅ Skills taxonomy loaded: 77 skills


## 4. Extract Skills & Experience (spaCy + NLTK)

In [ ]:
def extract_skills(text):
    doc = nlp(text)
    matches = matcher(doc)
    found = set()
    for match_id, start, end in matches:
        found.add(doc[start:end].text.lower())
    return sorted(found)

def extract_email(text):
    m = re.search(r"[\w\.-]+@[\w\.-]+\.\w+", text)
    return m.group(0) if m else None

def extract_phone(text):
    m = re.search(r"(\+?\d{1,3}[\s-]?)?\d{3,4}[\s-]?\d{6,7}", text)
    return m.group(0) if m else None

def extract_name(text):
    doc = nlp(text[:300])
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    return None

def extract_organizations(text):
    doc = nlp(text)
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    # dedupe, preserve order
    seen = set()
    result = []
    for o in orgs:
        if o.lower() not in seen:
            seen.add(o.lower())
            result.append(o)
    return result[:15]

def extract_dates(text):
    doc = nlp(text)
    return sorted(set(ent.text for ent in doc.ents if ent.label_ == "DATE"))

def extract_experience_years(text):
    matches = re.findall(r"(\d+(?:\.\d+)?)\+?\s*(?:years|yrs)\s*(?:of)?\s*experience", text.lower())
    if matches:
        return max(float(m) for m in matches)
    return None

def extract_education(text):
    edu_keywords = ["bachelor", "master", "bs ", "b.s.", "ms ", "m.s.", "phd", "bsc", "msc",
                     "university", "college", "degree", "cgpa", "gpa"]
    lines = text.split("\n")
    edu_lines = [l.strip() for l in lines if any(k in l.lower() for k in edu_keywords) and l.strip()]
    return edu_lines[:8]

def build_structured_resume(text):
    return {
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_skills(text),
        "organizations_mentioned": extract_organizations(text),
        "dates_mentioned": extract_dates(text),
        "years_of_experience": extract_experience_years(text),
        "education": extract_education(text),
    }

structured_resume = build_structured_resume(resume_text)
print(json.dumps(structured_resume, indent=2))


{
  "name": "Aleeha Fatima",
  "email": "fatimaaleeha054l@gmail.com",
  "phone": "+92 315 3937296",
  "skills": [
    "c++",
    "css",
    "data science",
    "html",
    "java",
    "machine learning",
    "problem solving",
    "python",
    "teamwork"
  ],
  "organizations_mentioned": [
    "fatimaaleeha054l@gmail.com",
    "CSS",
    "Problem Solving\n",
    "Programming Fundamentals\nData Structures\nObject Oriented Programming\nData Science\nMachine Learning\nDiscrete Mathematics",
    "4th Semester",
    "Data Science",
    "BS Mathematics with Data Science\nCOMSATS UNIVERSITY ISLAMABAD",
    "PUNJAB GROUP",
    "Symptom-Based Disease Prediction\nPython",
    "Naive Bayes",
    "\u203a",
    "UI"
  ],
  "dates_mentioned": [
    "2024",
    "315 3937296"
  ],
  "years_of_experience": null,
  "education": [
    "MS Office",
    "BS Mathematics with Data Science  \u00b7  4th Semester",
    "BS Mathematics with Data Science",
    "COMSATS UNIVERSITY ISLAMABAD, SAHIWAL CAMPUS",
    

## 5. Structured Resume Profile
Everything is shown right here in the notebook — no separate file downloads needed.

In [ ]:
from IPython.display import display, HTML, JSON

# Keep a copy in the Colab session (not downloaded) in case you want to reuse it later in this notebook
with open("structured_resume.json", "w") as f:
    json.dump(structured_resume, f, indent=2)

def render_profile_card(profile):
    skills_html = "".join(
        f'<span style="display:inline-block;background:#1F4E78;color:#fff;padding:4px 10px;'
        f'margin:3px;border-radius:14px;font-size:13px;">{s}</span>'
        for s in profile.get("skills", [])
    ) or "<i>No skills matched — try extending SKILLS_DB in Section 3.</i>"

    edu_html = "".join(f"<li>{e}</li>" for e in profile.get("education", [])) or "<li><i>Not detected</i></li>"
    org_html = ", ".join(profile.get("organizations_mentioned", [])) or "—"

    html = f"""
    <div style="font-family:Arial,sans-serif;border:1px solid #ddd;border-radius:10px;padding:18px 22px;max-width:720px;">
        <h2 style="margin:0 0 4px 0;color:#1F4E78;">{profile.get('name') or 'Name not detected'}</h2>
        <p style="margin:0 0 14px 0;color:#555;">
            {profile.get('email') or 'email not found'} &nbsp;|&nbsp; {profile.get('phone') or 'phone not found'}
        </p>
        <p><b>Years of experience:</b> {profile.get('years_of_experience') if profile.get('years_of_experience') is not None else 'Not stated'}</p>
        <p><b>Skills detected ({len(profile.get('skills', []))}):</b><br>{skills_html}</p>
        <p><b>Education:</b></p>
        <ul>{edu_html}</ul>
        <p><b>Organizations mentioned:</b> {org_html}</p>
    </div>
    """
    display(HTML(html))

render_profile_card(structured_resume)

print("\nFull structured JSON:")
display(JSON(structured_resume))



Full structured JSON:


<IPython.core.display.JSON object>

## 6. Demo Job Dataset
A small sample dataset to test the matching logic end-to-end before hitting a live API.

In [ ]:
demo_jobs = pd.DataFrame([
    {"title": "Data Science Intern", "company": "TechNova",
     "description": "Looking for a data science intern skilled in python, pandas, numpy, scikit-learn, "
                     "machine learning, and data visualization. Experience with SQL is a plus."},
    {"title": "NLP Engineer", "company": "LinguaAI",
     "description": "We need an NLP engineer with experience in spacy, nltk, deep learning, "
                     "natural language processing, and pytorch."},
    {"title": "Frontend Developer", "company": "PixelWorks",
     "description": "React and javascript developer needed, familiarity with css, html, and rest api integration."},
    {"title": "Data Analyst", "company": "InsightCorp",
     "description": "Analyze business data using sql, excel, tableau, power bi, and statistics. "
                     "Strong communication skills required."},
    {"title": "Machine Learning Intern", "company": "DeepMinds",
     "description": "Internship for students skilled in machine learning, tensorflow, keras, "
                     "neural networks, and python."},
    {"title": "Social Media Data Analyst", "company": "BuzzMetrics",
     "description": "Track engagement across social platforms using meta graph api, google analytics, "
                     "content strategy, and data analysis in python."},
    {"title": "Backend Developer", "company": "CloudBase",
     "description": "Node.js, django, docker, aws, and postgresql experience required for backend role."},
])
demo_jobs


,title,company,description
0,Data Science Intern,TechNova,Looking for a data science intern skilled in p...
1,NLP Engineer,LinguaAI,We need an NLP engineer with experience in spa...
2,Frontend Developer,PixelWorks,"React and javascript developer needed, familia..."
3,Data Analyst,InsightCorp,"Analyze business data using sql, excel, tablea..."
4,Machine Learning Intern,DeepMinds,Internship for students skilled in machine lea...
5,Social Media Data Analyst,BuzzMetrics,Track engagement across social platforms using...
6,Backend Developer,CloudBase,"Node.js, django, docker, aws, and postgresql e..."


## 7. Match Resume to Jobs — TF-IDF
Builds a TF-IDF vector space from the resume's skills/experience text and each job description, then ranks jobs by cosine similarity.

In [ ]:
def resume_to_text(structured):
    parts = structured.get("skills", []) + structured.get("organizations_mentioned", [])
    return " ".join(parts) if parts else resume_text

def tfidf_match(resume_struct, jobs_df, text_col="description", top_n=5):
    resume_query = resume_to_text(resume_struct)
    corpus = [resume_query] + jobs_df[text_col].tolist()
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(corpus)
    sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    ranked = jobs_df.copy()
    ranked["match_score"] = (sims * 100).round(1)
    return ranked.sort_values("match_score", ascending=False).head(top_n).reset_index(drop=True)

tfidf_results = tfidf_match(structured_resume, demo_jobs)
tfidf_results[["title", "company", "match_score"]]


,title,company,match_score
0,Data Science Intern,TechNova,30.5
1,Machine Learning Intern,DeepMinds,11.1
2,Social Media Data Analyst,BuzzMetrics,9.2
3,Frontend Developer,PixelWorks,8.3
4,Data Analyst,InsightCorp,6.7


## 8. Match Resume to Jobs — BERT (Semantic Similarity)
Uses `sentence-transformers` to capture meaning, not just keyword overlap — catches matches TF-IDF misses (e.g. "NLP" ↔ "natural language processing").

In [ ]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('all-MiniLM-L6-v2')

def bert_match(resume_struct, jobs_df, text_col="description", top_n=5):
    resume_query = resume_to_text(resume_struct)
    resume_emb = bert_model.encode([resume_query])
    job_embs = bert_model.encode(jobs_df[text_col].tolist())
    sims = cosine_similarity(resume_emb, job_embs).flatten()
    ranked = jobs_df.copy()
    ranked["match_score"] = (sims * 100).round(1)
    return ranked.sort_values("match_score", ascending=False).head(top_n).reset_index(drop=True)

bert_results = bert_match(structured_resume, demo_jobs)
bert_results[["title", "company", "match_score"]]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,title,company,match_score
0,Data Science Intern,TechNova,38.900002
1,Machine Learning Intern,DeepMinds,36.700001
2,Data Analyst,InsightCorp,31.600000
3,Frontend Developer,PixelWorks,28.299999
4,NLP Engineer,LinguaAI,26.400000


## 9. Live Job Search — Real Openings
Pulls **real, currently-live job postings** from the [Arbeitnow Job Board API](https://arbeitnow.com/api/job-board-api) — it's free and needs no API key, so it runs immediately.

> Want a different source? Swap the `fetch_live_jobs()` function below for another provider (e.g. Adzuna, RemoteOK, USAJobs) — most free job APIs return `title`/`company`/`description` fields you can map the same way. Paid APIs (Indeed, LinkedIn Jobs) need an API key you'd add in `HEADERS`.

In [ ]:
def fetch_live_jobs(query_skills, max_pages=2):
    """Fetch real live job listings from Arbeitnow (free, no API key required)."""
    all_jobs = []
    for page in range(1, max_pages + 1):
        try:
            resp = requests.get("https://www.arbeitnow.com/api/job-board-api", params={"page": page}, timeout=15)
            resp.raise_for_status()
            data = resp.json().get("data", [])
            if not data:
                break
            all_jobs.extend(data)
        except Exception as e:
            print(f"⚠️ Could not fetch page {page}: {e}")
            break
    if not all_jobs:
        return pd.DataFrame(columns=["title", "company", "description", "url"])
    jobs_df = pd.DataFrame(all_jobs)
    jobs_df = jobs_df.rename(columns={"company_name": "company"})
    keep_cols = [c for c in ["title", "company", "description", "url", "location", "remote"] if c in jobs_df.columns]
    return jobs_df[keep_cols]

live_jobs = fetch_live_jobs(structured_resume.get("skills", []), max_pages=2)
print(f"✅ Fetched {len(live_jobs)} live job postings")
live_jobs.head()


✅ Fetched 275 live job postings


,title,company,description,url,location,remote
0,Director of Product Management - Data Streaming,HiveMQ GmbH,"<p style=""min-height:1.5em""><em>HiveMQ is the ...",https://www.arbeitnow.com/jobs/companies/hivem...,Germany,False
1,Senior AI Fullstack Engineer – Cloud Applications,HiveMQ GmbH,"<p style=""min-height:1.5em""><em>HiveMQ is the ...",https://www.arbeitnow.com/jobs/companies/hivem...,Germany,False
2,Senior Software Engineer,HiveMQ GmbH,"<p style=""min-height:1.5em""><em>HiveMQ is the ...",https://www.arbeitnow.com/jobs/companies/hivem...,Germany,False
3,VP People (f/m/d),Lemon Markets,<h2><strong>About lemon.markets 🍋</strong></h2...,https://www.arbeitnow.com/jobs/companies/lemon...,Germany (Hybrid),False
4,Founder's Associate (f/m/d),Lemon Markets,<h2><strong>About lemon.markets</strong> 🍋</h2...,https://www.arbeitnow.com/jobs/companies/lemon...,Berlin Office,False


## 10. Rank Live Jobs Against Your Profile

In [ ]:
import re as _re

def clean_html(text):
    return _re.sub("<[^<]+?>", " ", str(text))

if len(live_jobs) > 0:
    live_jobs_clean = live_jobs.copy()
    live_jobs_clean["description"] = live_jobs_clean["description"].apply(clean_html)

    print("=== TF-IDF top matches (live jobs) ===")
    live_tfidf = tfidf_match(structured_resume, live_jobs_clean, top_n=10)
    display_cols = [c for c in ["title", "company", "match_score", "url"] if c in live_tfidf.columns]
    display(live_tfidf[display_cols])

    print("\n=== BERT top matches (live jobs) ===")
    live_bert = bert_match(structured_resume, live_jobs_clean, top_n=10)
    display(live_bert[display_cols])
else:
    print("No live jobs fetched — check your internet connection or try increasing max_pages.")


=== TF-IDF top matches (live jobs) ===


,title,company,match_score,url
0,"Senior Data Scientist, Growth",Arq,15.5,https://www.arbeitnow.co.uk/jobs/companies/arq...
1,"Senior Data Scientist, FinCrime",Arq,14.4,https://www.arbeitnow.co.uk/jobs/companies/arq...
2,Data Scientist,Fundingcircle,13.5,https://www.arbeitnow.co.uk/jobs/companies/fun...
3,Data Analyst,Fundingcircle,12.2,https://www.arbeitnow.co.uk/jobs/companies/fun...
4,"VP, Data",Fundingcircle,11.5,https://www.arbeitnow.co.uk/jobs/companies/fun...
5,Senior Engineer,Fundingcircle,10.8,https://www.arbeitnow.co.uk/jobs/companies/fun...
6,Data Quality Engineer,Fundingcircle,10.0,https://www.arbeitnow.co.uk/jobs/companies/fun...
7,"Director, Marketing Data Science & Analytics",Omaze,9.6,https://www.arbeitnow.co.uk/jobs/companies/oma...
8,Senior Software Engineer (Java),Arq,9.1,https://www.arbeitnow.co.uk/jobs/companies/arq...
9,Software Engineer (Java),Arq,9.1,https://www.arbeitnow.co.uk/jobs/companies/arq...



=== BERT top matches (live jobs) ===


,title,company,match_score,url
0,"Senior Data Scientist, FinCrime",Arq,34.200001,https://www.arbeitnow.co.uk/jobs/companies/arq...
1,Senior Data Scientist / ML Engineer (Forecasti...,Gt Hq,30.500000,https://www.arbeitnow.co.uk/jobs/companies/gt-...
2,Product AI Agent Builder,Moss,27.000000,https://www.arbeitnow.com/jobs/companies/moss/...
3,Senior Data Scientist (f/m/d),Moss,26.500000,https://www.arbeitnow.com/jobs/companies/moss/...
4,Partner Success Manager,Litmus,22.900000,https://www.arbeitnow.com/jobs/companies/litmu...
5,Sales Engineer,Litmus,22.900000,https://www.arbeitnow.com/jobs/companies/litmu...
6,Customer Success Application Engineer (EMEA),Litmus,22.900000,https://www.arbeitnow.com/jobs/companies/litmu...
7,Principal Thermal-Mechanical Engineer - (Munic...,Vinci4D,22.600000,https://www.arbeitnow.com/jobs/companies/vinci...
8,Senior Product Manager,Kuro Technology,21.799999,https://www.arbeitnow.com/jobs/companies/kuro-...
9,(Senior) Backend Engineer (f/m/d),bunch,21.700001,https://www.arbeitnow.com/jobs/companies/bunch...


## 11. Final Summary Dashboard
Everything — the structured profile, demo matches, and live matches — is rendered inline below. Nothing needs to be downloaded separately.

In [ ]:
from IPython.display import display, HTML

def render_matches_table(title, df):
    if df is None or len(df) == 0:
        display(HTML(f"<h3 style='color:#1F4E78'>{title}</h3><p><i>No results.</i></p>"))
        return
    cols = [c for c in ["title", "company", "match_score", "url"] if c in df.columns]
    rows_html = ""
    for _, row in df[cols].iterrows():
        cells = "".join(f"<td style='padding:6px 10px;border-bottom:1px solid #eee;'>{row[c]}</td>" for c in cols)
        rows_html += f"<tr>{cells}</tr>"
    header_html = "".join(f"<th style='padding:6px 10px;text-align:left;background:#1F4E78;color:#fff;'>{c.replace('_',' ').title()}</th>" for c in cols)
    html = f"""
    <h3 style="color:#1F4E78;margin-bottom:6px;">{title}</h3>
    <table style="border-collapse:collapse;width:100%;max-width:800px;font-family:Arial,sans-serif;font-size:14px;">
        <tr>{header_html}</tr>
        {rows_html}
    </table>
    """
    display(HTML(html))

display(HTML("<h1 style='color:#1F4E78;'>📊 Resume Screening Summary</h1>"))
render_profile_card(structured_resume)

render_matches_table("Top Demo-Dataset Matches — TF-IDF", tfidf_results)
render_matches_table("Top Demo-Dataset Matches — BERT", bert_results)

if len(live_jobs) > 0:
    render_matches_table("Top Live Job Matches — TF-IDF", live_tfidf)
    render_matches_table("Top Live Job Matches — BERT", live_bert)
else:
    display(HTML("<p><i>No live jobs were fetched in Section 9 — re-run that cell to populate this section.</i></p>"))

print("\n✅ Everything above is generated fresh each run — no separate file download needed.")


Title,Company,Match Score
Data Science Intern,TechNova,30.5
Machine Learning Intern,DeepMinds,11.1
Social Media Data Analyst,BuzzMetrics,9.2
Frontend Developer,PixelWorks,8.3
Data Analyst,InsightCorp,6.7


Title,Company,Match Score
Data Science Intern,TechNova,38.900001525878906
Machine Learning Intern,DeepMinds,36.70000076293945
Data Analyst,InsightCorp,31.600000381469727
Frontend Developer,PixelWorks,28.299999237060547
NLP Engineer,LinguaAI,26.399999618530273


Title,Company,Match Score,Url
"Senior Data Scientist, Growth",Arq,15.5,https://www.arbeitnow.co.uk/jobs/companies/arq/senior-data-scientist-growth-london-430160
"Senior Data Scientist, FinCrime",Arq,14.4,https://www.arbeitnow.co.uk/jobs/companies/arq/senior-data-scientist-fincrime-london-7159
Data Scientist,Fundingcircle,13.5,https://www.arbeitnow.co.uk/jobs/companies/fundingcircle/data-scientist-london-290249
Data Analyst,Fundingcircle,12.2,https://www.arbeitnow.co.uk/jobs/companies/fundingcircle/data-analyst-london-53897
"VP, Data",Fundingcircle,11.5,https://www.arbeitnow.co.uk/jobs/companies/fundingcircle/vp-data-london-415901
Senior Engineer,Fundingcircle,10.8,https://www.arbeitnow.co.uk/jobs/companies/fundingcircle/senior-engineer-london-409751
Data Quality Engineer,Fundingcircle,10.0,https://www.arbeitnow.co.uk/jobs/companies/fundingcircle/data-quality-engineer-london-435983
"Director, Marketing Data Science & Analytics",Omaze,9.6,https://www.arbeitnow.co.uk/jobs/companies/omaze/director-marketing-data-science-analytics-london-42691
Senior Software Engineer (Java),Arq,9.1,https://www.arbeitnow.co.uk/jobs/companies/arq/senior-software-engineer-java-london-229213
Software Engineer (Java),Arq,9.1,https://www.arbeitnow.co.uk/jobs/companies/arq/software-engineer-java-london-291023


Title,Company,Match Score,Url
"Senior Data Scientist, FinCrime",Arq,34.20000076293945,https://www.arbeitnow.co.uk/jobs/companies/arq/senior-data-scientist-fincrime-london-7159
Senior Data Scientist / ML Engineer (Forecasting) | NDA,Gt Hq,30.5,https://www.arbeitnow.co.uk/jobs/companies/gt-hq/remote-senior-data-scientist-ml-engineer-forecasting-nda-178267
Product AI Agent Builder,Moss,27.0,https://www.arbeitnow.com/jobs/companies/moss/product-ai-agent-builder-berlin-133601
Senior Data Scientist (f/m/d),Moss,26.5,https://www.arbeitnow.com/jobs/companies/moss/senior-data-scientist-berlin-222706
Partner Success Manager,Litmus,22.899999618530273,https://www.arbeitnow.com/jobs/companies/litmus/partner-success-manager-munchen-44018
Sales Engineer,Litmus,22.899999618530273,https://www.arbeitnow.com/jobs/companies/litmus/sales-engineer-munchen-203583
Customer Success Application Engineer (EMEA),Litmus,22.899999618530273,https://www.arbeitnow.com/jobs/companies/litmus/customer-success-application-engineer-emea-munchen-26188
"Principal Thermal-Mechanical Engineer - (Munich, Germany)",Vinci4D,22.600000381469727,https://www.arbeitnow.com/jobs/companies/vinci4d/principal-thermal-mechanical-engineer-munich-germany-338918
Senior Product Manager,Kuro Technology,21.799999237060547,https://www.arbeitnow.com/jobs/companies/kuro-technology/senior-product-manager-berlin-402032
(Senior) Backend Engineer (f/m/d),bunch,21.700000762939453,https://www.arbeitnow.com/jobs/companies/bunch/senior-backend-engineer-berlin-495275



✅ Everything above is generated fresh each run — no separate file download needed.
